The following script restructures the raw eye-tracking dataset into
one file per participant, making it easier to analyze individual data.

This script does the following.
- reads all CSV files from ../data/raw
- identifies the participant column case-insensitively
- converts participant values to strings
- splits rows into per-participant CSVs in ../data/by_participant
- removes those same rows from the original files

In [1]:
import pandas as pd
from pathlib import Path

# --- Paths ---
raw_dir = Path("../data/rawdata")
out_dir = Path("../data/by_participant")
out_dir.mkdir(parents=True, exist_ok=True)

# Participants of interest: "1"..."59" (as strings)
participants = [str(i) for i in range(1, 60)]

# Accumulator for removed rows per participant
per_participant = {p: [] for p in participants}

# --- Process each raw data file once ---
for i in range(1, 26):
    path = raw_dir / f"{i}.csv"
    print(f"Processing file {path}...")

    df = pd.read_csv(path, low_memory=False)

    # Find participant column (case-insensitive)
    cols_lower = {c.lower(): c for c in df.columns}
    part_col = cols_lower.get("participant")

    if part_col is None:
        raise ValueError(f"No 'participant' column (any case) found in {path}")

    # Make sure we compare as strings
    part_values = df[part_col].astype(str)

    # Mask for rows we want to extract (participants 1–59)
    mask = part_values.isin(participants)

    # Subset with rows to remove from this file
    subset = df[mask]

    if not subset.empty:
        print(f"  → {len(subset)} rows with participant in 1–59")

        # Group by participant value and store in accumulator
        for value, group in subset.groupby(part_values[mask]):
            per_participant[value].append(group)

    # Keep only rows NOT in participants 1–59
    df_filtered = df[~mask]

    # Only rewrite file if something changed
    if len(df_filtered) != len(df):
        df_filtered.to_csv(path, index=False)
        print(f"  → wrote filtered file (removed {len(df) - len(df_filtered)} rows)")

# --- Write one output file per participant ---
for p, chunks in per_participant.items():
    if not chunks:
        continue  # no rows for this participant in any file

    all_removed = pd.concat(chunks, ignore_index=True, sort=False)
    out_path = out_dir / f"participant_{p}.csv"
    all_removed.to_csv(out_path, index=False)
    print(f"Wrote {len(all_removed)} rows to {out_path}")

Processing file ../data/rawdata/1.csv...
Processing file ../data/rawdata/2.csv...
Processing file ../data/rawdata/3.csv...
Processing file ../data/rawdata/4.csv...
Processing file ../data/rawdata/5.csv...
Processing file ../data/rawdata/6.csv...
Processing file ../data/rawdata/7.csv...
Processing file ../data/rawdata/8.csv...
Processing file ../data/rawdata/9.csv...
Processing file ../data/rawdata/10.csv...
Processing file ../data/rawdata/11.csv...
Processing file ../data/rawdata/12.csv...
Processing file ../data/rawdata/13.csv...
Processing file ../data/rawdata/14.csv...
Processing file ../data/rawdata/15.csv...
Processing file ../data/rawdata/16.csv...
Processing file ../data/rawdata/17.csv...
Processing file ../data/rawdata/18.csv...
Processing file ../data/rawdata/19.csv...
Processing file ../data/rawdata/20.csv...
Processing file ../data/rawdata/21.csv...
Processing file ../data/rawdata/22.csv...
Processing file ../data/rawdata/23.csv...
Processing file ../data/rawdata/24.csv...
P